# EDA — Personality Type Prediction

**Project step 1 of 3:** `eda.ipynb` → `modeling.ipynb` → `app.py`

This notebook builds on Live Session 1 (`JT_LS1_data_and_pipeline.ipynb`). It keeps the
data loading and the quick look from LS1 and adds the checks the submission asks for:

1. load `data/data.csv` and check its shape and types,
2. look for **missing** and **impossible** values,
3. check how **balanced** the target classes are,
4. look at the **distribution of the personality items**,
5. apply the cleaning decisions in **one place**,
6. save the result as **`data/data_clean.csv`** — the handoff to `modeling.ipynb`.

> The preprocessing **pipeline** (imputing, scaling, one-hot encoding) will live in `modeling.ipynb`. This notebook only *fixes* the data; it does not transform it for a model.

## 1. Setup, package verification and load the data

In [ ]:
# Setup packages
import subprocess
import sys
from pathlib import Path

for pkg in ["gdown", "ipywidgets"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", pkg],
            check=True
        )

# Imports
import gdown
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
import textwrap

from IPython.display import HTML, display


# Data path
FILE_ID = "1lgycXDQapixkoWFxIg48qaYcrNl6uSZM"

DATA_DIR = Path("DATA")
DATA_DIR.mkdir(exist_ok=True)
DATA_FILE = DATA_DIR / "data.csv"


# Setup progress
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Setup:",
    bar_style="info"
)

status = widgets.HTML(value="Starting setup...")
output = widgets.Output()

with output:
    display(progress, status)

display(output)


# Check dataset
progress.value = 20
status.value = "Checking dataset..."

if DATA_FILE.exists():
    progress.value = 100
    progress.bar_style = "success"
    status.value = "✓ Dataset already available."

else:
    progress.value = 40
    status.value = "Downloading dataset..."

    url = f"https://drive.google.com/uc?id={FILE_ID}"
    gdown.download(url, output=str(DATA_FILE), quiet=True)

    if not DATA_FILE.exists():
        raise FileNotFoundError(
            f"Download failed. File not found: {DATA_FILE}"
        )

    progress.value = 100
    progress.bar_style = "success"
    status.value = "✓ Download complete."

In [ ]:
# Constants & Configurations
MIN_AGE = 1
MAX_AGE = 100
REFERENCE_YEAR = 2017
MIN_BIRTH_YEAR = REFERENCE_YEAR - MAX_AGE + 1

TRAITS = {
    "O": "Openness to Experience",
    "C": "Conscientiousness",
    "E": "Extraversion",
    "A": "Agreeableness",
    "N": "Neuroticism"
}

ocean_order = list(TRAITS.values())

# Fixed color mapping for consistent category colors across all plots
TARGET_COLORS = {
    "Openness":         "#6a4c93",  # O - Harmonious violet/purple (matching Mako)
    "Resilient":        "#e07a5f",  # C - Muted terracotta/coral
    "Overcontroller":   "#81b29a",  # E - Sage/mint green
    "Undercontroller":  "#f2cc8f",  # A - Warm ochre yellow
    "Moderate":         "#2b5c6f"   # N - Mako petroleum blue (aligned with primary theme)
}

target_order = list(TARGET_COLORS.keys())

# Harmonized theme matching TARGET_COLORS and the Mako palette
THEME = {
    "primary":          "#2b5c6f",  # Main petroleum blue for titles & headers
    "success_bg":       "#edf7f4",  # Soft mint/sage background
    "success_border":   "#81b29a",  # Sage/mint green border (matches Overcontroller)
    "danger_bg":        "#fdf2f2",  # Soft coral/red background
    "danger_border":    "#e07a5f",  # Terracotta/coral border (matches Resilient)
    "palette":          "Blues"
}

In [ ]:
# CSV Setup & Loading
df = pd.read_csv(DATA_FILE)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)

question_columns = [
    'N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10',
    'E1', 'E3', 'E4', 'E5', 'E7', 'E9', 'E10', 'C4', 'A4'
]

display(HTML(f"""
<div style="display:inline-block; padding:10px 14px; margin:8px 0; 
            background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; 
            border-radius:5px; font-size:14px;">
    <b>Initial data:</b> {len(df):,} rows · {df.shape[1]} columns
</div>
"""))

### 1.5 Checking the existing columns and initial data
##### Structure, data types and missing values

In [ ]:
display(HTML("<h4 style='margin:16px 0 8px;'>First 5 rows (head)</h4>"))
display(df.head())

display(HTML("<h4 style='margin:16px 0 8px;'>Last 5 rows (tail)</h4>"))
display(df.tail())

display(HTML("<h4 style='margin:16px 0 8px;'>Summary statistics (numerical columns)</h4>"))
display(df.describe())

column_info = pd.DataFrame({
    "Data type": df.dtypes.astype(str),
    "Non-null": df.notna().sum(),
    "Missing": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(2)
})
column_info["Status"] = np.where(column_info["Missing"] > 0, "⚠ Missing values", "✓ Complete")

display(HTML("<h4 style='margin:16px 0 8px;'>Column overview & missing values</h4>"))
display(column_info)

display(HTML("<h4 style='margin:16px 0 8px;'>Categorical & key demographic distributions</h4>"))
html = '<div style="display:flex; flex-wrap:wrap; gap:18px; align-items:flex-start;">'
for col in ["age", "gender", "hand", "target"]:
    table = df[col].value_counts(dropna=False).sort_index().to_frame("Count").to_html()
    html += f"""
    <div style="min-width:150px; padding:10px; background:#f8fafc; border:1px solid #d9e2ec; border-radius:6px;">
        <div style="margin-bottom:8px; color:{THEME['primary']}; font-weight:700;">{col}</div>
        {table}
    </div>
    """
html += '</div>'
display(HTML(html))

>**One row = one person**
>
> Before modelling we check the column types, gaps and duplicates.

# Exploratory Data Analysis (EDA)
 
##  2. Target Analysis

In [ ]:
# Numeric personality-question columns (every column except the demographics and target)
item_cols = [
    c for c in df.columns
    if c not in ["age", "gender", "hand", "target"]
    and pd.api.types.is_numeric_dtype(df[c])
]

# Only existing targets, in the fixed TARGET_COLORS order
target_order = [
    target for target in TARGET_COLORS
    if target in df["target"].dropna().unique()
]

# Mean response per target, in the same order as the bar chart
heatmap_data = (
    df.groupby("target", observed=True)[item_cols]
      .mean()
      .reindex(target_order)
)

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(20, 5),
    gridspec_kw={"width_ratios": [1, 2.5]}
)

# What are we predicting? The personality type.
# 1. Target distribution
total = len(df)

sns.countplot(
    data=df,
    x="target",
    hue="target",
    order=target_order,
    hue_order=target_order,
    palette=TARGET_COLORS,
    legend=False,
    ax=ax1
)

for container in ax1.containers:
    ax1.bar_label(
        container,
        labels=[
            f"{bar.get_height():,}\n({bar.get_height() / total:.1%})"
            if bar.get_height() > 0 else ""
            for bar in container
        ]
    )

ax1.set_title(
    "Target Distribution",
    fontsize=13,
    pad=12,
    color=THEME["primary"],
    weight="bold"
)
ax1.set_xlabel("")
ax1.set_ylabel("Count", fontsize=11)

# 2. Mean response score to each question per personality type. This is the kind of signal the model will learn from.
sns.heatmap(
    heatmap_data,
    cmap=THEME["palette"],  # "Blues"
    vmin=1,
    vmax=5,
    ax=ax2,
    linewidths=0.5,
    cbar_kws={"label": "Mean Score"}
)

ax2.set_title(
    "Average Response per Personality Type",
    fontsize=13,
    pad=12,
    color=THEME["primary"],
    weight="bold"
)
ax2.set_xlabel("Question Items", fontsize=11)
ax2.set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# Data cleaning logic
age_num = pd.to_numeric(df['age'], errors='coerce')
missing_age = age_num.isna()
three_digit_age = age_num.between(MAX_AGE + 1, 999, inclusive="both")
valid_age = age_num.between(MIN_AGE, MAX_AGE, inclusive="both")
birth_year = age_num.between(MIN_BIRTH_YEAR, REFERENCE_YEAR, inclusive='both')
invalid_age = ~(birth_year | valid_age) & ~missing_age

age_clean = age_num.copy()
age_clean.loc[birth_year] = REFERENCE_YEAR - age_num.loc[birth_year]

def missing_or_zero(series):
    normalized = series.astype('string').str.strip().str.lower()
    return series.isna() | normalized.isin({'0', 'none', 'null', 'nan', ''})

missing_gender = missing_or_zero(df['gender'])
missing_hand = missing_or_zero(df['hand'])

# Zero values in question items
question_zero = pd.Series(False, index=df.index)
for col in question_columns:
    if col in df.columns:
        question_zero |= pd.to_numeric(df[col], errors='coerce').eq(0)

zero_rows = df.loc[question_zero].copy()
display(HTML(f"<h4>Rows with at least one zero in the questions ({len(zero_rows):,})</h4>"))
display(zero_rows)

In [ ]:
# Stepwise cleaning, column reordering, and cumulative data loss tracking
df_work = df.copy()
df_work["age"] = age_clean

total_rows = len(df_work)
keep = pd.Series(True, index=df_work.index)

steps = [
    ("age missing", missing_age),
    ("age three-digit", three_digit_age),
    ("age otherwise invalid", invalid_age),
    ("gender missing or 0", missing_gender),
    ("hand missing or 0", missing_hand),
    ("0 in question items", question_zero),
]

results = []
for name, criterion in steps:
    removed_now = keep & criterion
    removed_count = removed_now.sum()
    keep &= ~criterion

    remaining = keep.sum()
    removed_pct = removed_count / total_rows * 100
    cumulative_removed = total_rows - remaining
    cumulative_pct = cumulative_removed / total_rows * 100

    results.append({
        "Step": name,
        "Removed in this step": removed_count,
        "Removed (%)": round(removed_pct, 3),
        "Remaining rows": remaining,
        "Cumulative removed": cumulative_removed,
        "Cumulative data loss (%)": round(cumulative_pct, 3),
    })

df_clean = df_work.loc[keep].copy()
df_clean["age"] = df_clean["age"].astype("Int64")

# Reorder columns: place 'age' directly before 'gender'
cols = [c for c in df_clean.columns if c != "age"]
cols.insert(cols.index("gender"), "age")
df_clean = df_clean[cols]

loss_table = pd.DataFrame(results)
display(loss_table)

rows_deleted = len(df) - len(df_clean)
display(HTML(f"""
<div style="display:inline-block; padding:10px 14px; margin:8px 0; 
            background:{THEME['danger_bg']}; border-left:4px solid {THEME['danger_border']}; 
            border-radius:5px; font-size:14px;">
    <b>Initial data:</b> {len(df):,} rows · {df.shape[1]} columns<br>
    <b>Cleaned data:</b> {len(df_clean):,} rows · {df_clean.shape[1]} columns<br>
    <b>Deleted data:</b> {rows_deleted:,} rows ({rows_deleted / len(df) * 100:.3f} %)
</div>
"""))

# Save clean dataset
df_clean.to_csv(DATA_DIR / "data_clean.csv", index=False)

# Data Exploration 

### Correlations between the questions

The column names tell us which OCEAN trait a question belongs to:

> O = Openness<br>
> C = Conscientiousness<br>
> E = Extraversion<br>
> A = Agreeableness<br>
> N = Neuroticism<br>

The 19 items are not spread evenly across the traits. Ten measure Neuroticism and seven Extraversion, but Conscientiousness and Agreeableness have only one item each (C4, A4). Since Undercontroller is defined precisely by C and A, the information needed to spot that type is largely missing from the data. This explains directly why the model only catches ~20% of Undercontrollers while getting most of the Resilients.

In [ ]:
table_html = (
    pd.Series([c[0] for c in item_cols])
    .map(TRAITS)
    .value_counts()
    .reindex(ocean_order, fill_value=0)
    .to_frame("Count")
    .to_html(classes="table table-striped table-hover", border=0)
)

display(
    HTML(
        f"<h4 style='margin:16px 0 8px; color:{THEME['primary']};'>Question Items per OCEAN Dimension</h4>"
    )
)
display(HTML(table_html))

### Question Item Distributions by Target

A `countplot` per column, split by `target`. For each answer (1–5) we see how the four personality types are distributed. Where the colored blocks shift from left to right between the types, that question separates them — that is the signal the model will learn.

In [ ]:
item_labels = {
    "N1":  "I get stressed out easily.",
    "N2":  "I am relaxed most of the time.",
    "N3":  "I worry about things.",
    "N4":  "I seldom feel blue.",
    "N5":  "I am easily disturbed.",
    "N6":  "I get upset easily.",
    "N7":  "I change my mood a lot.",
    "N8":  "I have frequent mood swings.",
    "N9":  "I get irritated easily.",
    "N10": "I often feel blue.",
    "E1":  "I am the life of the party.",
    "E3":  "I feel comfortable around people.",
    "E4":  "I keep in the background.",
    "E5":  "I start conversations.",
    "E7":  "I talk to a lot of different people at parties.",
    "E9":  "I don't mind being the center of attention.",
    "E10": "I am quiet around strangers.",
    "C4":  "I make a mess of things.",
    "A4":  "I sympathize with others' feelings.",
}

target_order = [
    target for target in TARGET_COLORS
    if target in df_clean["target"].dropna().unique()
]

# Plot layout
n = len(item_cols)
ncols = 4
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(20, 4 * nrows),
    sharey=True
)

for i, (ax, col) in enumerate(zip(axes.flat, item_cols)):
    sns.countplot(
        data=df_clean,
        x=col,
        hue="target",
        hue_order=target_order,
        palette=TARGET_COLORS,
        ax=ax,
        legend=(i == 0)
    )

    title = f"{col} — {item_labels.get(col, '')}"
    ax.set_title(
        "\n".join(textwrap.wrap(title, 34)),
        fontsize=10
    )
    ax.set_xlabel("")
    ax.set_ylabel("")

# Remove empty plots
for ax in axes.flat[n:]:
    ax.remove()

# Shared legend
handles, labels = axes.flat[0].get_legend_handles_labels()
axes.flat[0].get_legend().remove()

fig.legend(
    handles,
    labels,
    title="Personality type",
    loc="lower center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=4
)

fig.suptitle(
    "Answer distribution per question, split by personality type",
    fontsize=15,
    y=1.0
)

plt.tight_layout(rect=[0, 0.03, 1, 0.985])
plt.show()

### 2.2 The question scores

The instructions say every answer is on a 1–5 scale. Let's check if this is really true.

In [ ]:
df[item_cols].describe().T[["mean", "std", "min", "max"]].round(2)

Are any answers outside the 1-5 scale?

In [ ]:
out_of_range = (df[item_cols] < 1) | (df[item_cols] > 5)
print("Answers outside 1-5:", out_of_range.sum().sum())
print("Rows affected:     ", out_of_range.any(axis=1).sum())
df[out_of_range.any(axis=1)]

A total of 19 columns (in one row) have at least one 0 answer that is out of scale. One person answered all out of scale and should be removed along with the other blank rows later.

How are the answers distributed?

Share of people giving each answer, per question.

In [ ]:
# Percentage share of answers (1–5) per question item
answer_share = (
    df_clean[item_cols]
    .apply(lambda col: col.value_counts(normalize=True))
    .reindex([1, 2, 3, 4, 5])
    .T
)

plt.figure(figsize=(9, 8))

max_share = answer_share.to_numpy().max()

sns.heatmap(
    answer_share,
    annot=True,
    fmt=".0%",
    cmap=THEME["palette"],
    vmin=0,
    vmax=max_share,
    linewidths=0.5,
    cbar_kws={
        "label": "Share of Responses",
        "format": mtick.PercentFormatter(xmax=1)
    }
)

plt.title(
    "Response Distribution (1 to 5) per Question Item",
    fontsize=14,
    pad=15,
    color=THEME["primary"],
    weight="bold"
)
plt.xlabel("Answer Score (1 = Disagree, 5 = Agree)", fontsize=11, labelpad=10)
plt.ylabel("Question Items", fontsize=11, labelpad=10)
plt.tight_layout()
plt.show()

### 2.3 Correlations between the questions

The column names tell us which OCEAN trait a question belongs to (N = Neuroticism, E = Extraversion,
C = Conscientiousness, A = Agreeableness).

The 19 items are not spread evenly across the traits. Ten measure Neuroticism and seven Extraversion, but Conscientiousness and Agreeableness have only one item each (C4, A4). Since Undercontroller is defined precisely by C and A, the information needed to spot that type is largely missing from the data. This explains directly why the model only catches ~20% of Undercontrollers while getting most of the Resilients.

In [ ]:
# Count of questions per trait
pd.Series([c[0] for c in item_cols]).value_counts().rename({"N": "Neuroticism", "E": "Extraversion", "C": "Conscientiousness", "A": "Agreeableness"})

In [ ]:
# Select only numeric question item columns (or all numeric columns)
numeric_df = df_clean.select_dtypes(include=[np.number])

# Compute correlation matrix
corr = numeric_df.corr()

# 1. Heatmap Plot
plt.figure(figsize=(16, 12))
sns.heatmap(
    corr,
    cmap="coolwarm", 
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson Correlation Coefficient"}
)
plt.title("Correlation Matrix (Numeric Features & Question Items)", fontsize=14, pad=15)
plt.show()

# 2. Extract unique correlation pairs (excluding self-correlations)
corr_matrix = corr.abs()
upper_tri = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper_tri.unstack().dropna()

top_pos = (
    pairs.sort_values(ascending=False)
    .head(5)
    .round(2)
    .to_frame("Correlation")
    .to_html(classes="table table-hover")
)

top_neg = (
    pairs.sort_values(ascending=True)
    .head(5)
    .round(2)
    .to_frame("Correlation")
    .to_html(classes="table table-hover")
)

# 3. HTML Summary Display matched with THEME
html = f"""
<div style="display:flex; gap:30px; align-items:flex-start; margin-top:10px;">
    <div style="min-width:240px; padding:14px; background:#f8fafc; border:1px solid #d9e2ec; 
                border-top:4px solid {THEME['success_border']}; border-radius:6px;">
        <h4 style="margin:0 0 10px 0; color:{THEME['primary']}; font-weight:700;">
            Strongest Positive Pairs
        </h4>
        {top_pos}
    </div>
    <div style="min-width:240px; padding:14px; background:#f8fafc; border:1px solid #d9e2ec; 
                border-top:4px solid {THEME['danger_border']}; border-radius:6px;">
        <h4 style="margin:0 0 10px 0; color:{THEME['primary']}; font-weight:700;">
            Strongest Negative Pairs
        </h4>
        {top_neg}
    </div>
</div>
"""

display(HTML(html))

### 2.4 Age Analysis — looking for outliers

Age is messy. The maximum is 990,000,000. 83 values are over 100, and several look like birth years that were typed in (1961, 1974, 1984…). The minimum is 13, and about 1,450 people are under 16. We need a rule here, for example treating anything outside a plausible range (say 13–100) as missing or imputing the median.

Age is strongly skewed toward young respondents (73 % under 30) in every personality type.

Resilient people with the widest spread (Q3 at 36 vs. 25) are slightly older (median 24) and Undercontrollers slightly younger (median 20), but the distributions overlap so heavily that **age alone barely separates the classes — the predictive signal comes from the questionnaire answers.**

In [ ]:
# Dynamically filter target_order to only include targets that actually exist in the cleaned dataset
target_order = [t for t in TARGET_COLORS.keys() if t in df_clean["target"].unique()]

min_age = int(df_clean["age"].min())
max_age = int(df_clean["age"].max())

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(15, 11))

# 1. Top Left: Age distribution per personality type
sns.boxenplot(
    data=df_clean,
    x="target",
    y="age",
    order=target_order,
    palette=TARGET_COLORS,
    hue="target",
    legend=False,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Age per Personality Type")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Age")

# 2. Top Right: Age histogram by target
sns.histplot(
    data=df_clean,
    x="age",
    hue="target",
    hue_order=target_order,
    palette=TARGET_COLORS,
    bins=40,
    element="step",
    legend=False,
    ax=axes[0, 1]
)
axes[0, 1].set_title(f"Age Distribution (Ages {min_age} till {max_age})")
axes[0, 1].set_xlabel("Age")
axes[0, 1].set_ylabel("Count")

# 3. Bottom Left: Gender distribution
sns.countplot(
    data=df_clean,
    x="gender",
    hue="target",
    hue_order=target_order,
    palette=TARGET_COLORS,
    legend=False,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Gender Distribution by Personality Type")
axes[1, 0].set_xlabel("Gender")
axes[1, 0].set_ylabel("Count")

# 4. Bottom Right: Handedness distribution (Left and Right swapped: Left, Both, Right)
sns.countplot(
    data=df_clean,
    x="hand",
    order=["Left", "Both", "Right"],
    hue="target",
    hue_order=target_order,
    palette=TARGET_COLORS,
    ax=axes[1, 1]
)
axes[1, 1].set_title("Handedness Distribution by Personality Type")
axes[1, 1].set_xlabel("Handedness")
axes[1, 1].set_ylabel("Count")

# Extract legend handles from the last subplot and place them centrally below the figure
handles, labels = axes[1, 1].get_legend_handles_labels()
axes[1, 1].get_legend().remove()

fig.legend(
    handles=handles,
    labels=labels,
    title="Personality Type",
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=len(target_order),
    frameon=True
)

plt.tight_layout()
plt.show()

### 2.5 Gender and hand - missing categories

124 missing cells across 120 rows (gender 24, hand 100 — so 4 people are missing both)

In [ ]:
for col in ["gender", "hand"]:
    print(df[col].value_counts(dropna=False), "\n")
display(df)

### 2.6 Key findings from the EDA

- **Size:** 19,719 people, 23 columns — 19 question scores, `age`, `gender`, `hand` and the `target`.
- **Imbalanced classes:** Moderate ≈ 43 %, Resilient ≈ 31 %, Overcontroller ≈ 14 %, Undercontroller ≈ 12 %.
  A model that always says "Moderate" would already get ~43 % accuracy, so accuracy alone can be misleading.
- **The questions carry clear signal:** the Neuroticism questions (N…) separate Resilient (low answers) from Overcontroller (high answers). Some Extraversion questions correlate *negatively* with others —
  these are reverse-worded questions, which is fine for the models.
- **Invalid answers:** one row has for numerical questions all `0` except for age for all questions. `0` is not on the 1–5 scale (in the original Kaggle data it means "not answered"), so this row is dropped.
- **Age has impossible values:** a maximum of 999,000,000, 83 ages above 100 — 73 of the 83 implausible ages look like birth years that were typed in; here we converted them the survey year 2017 as mean of 2016-2018.
- **Missing values:** `gender` (24) and `hand` (100) — small, they are removed and the pipeline's imputer can handle them in the future.
- **Gender** shows some differences between types (e.g. more Resilient among men), **hand** hardly any.

## Cleaned the data before modelling

The pipeline fills gaps, but it cannot tell that an age of 999,000,000 is wrong. So we remove the problems found above **before** splitting into X and y. Every row is deleted where:

- all numerical columns have the exact same answer (once)
- an answer is `0`, because that is outside the allowed range of 1–5
- `hand` is empty
- `gender` is empty
- `age` has impossible values and cannot be converted from a birth year (given data set is from 2017 as base year)

In the Streamlit app the inputs are restricted (sliders 1–5, age 13–100, required dropdowns), so a new user cannot enter these kinds of values.